<a href="https://colab.research.google.com/github/moi-matt/pdf_syntthesizor_vllm/blob/main/pdf_synth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthsize a pdf document with a LLM

## Import des librairies utiles

In [1]:
#check nvidia
!nvidia-smi

Sun Jul 19 20:47:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install vllm

In [11]:
!pip install pypdf
!pip install transformers
!pip install langchain
!pip install langchain_core
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.3 MB/s eta 0:00:00


In [2]:
import requests
from pypdf import PdfReader
import tiktoken
from transformers import AutoTokenizer
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

## Téléchargement du document et extraction du texte

In [3]:
# Download the document
doc = requests.get("https://arxiv.org/pdf/2311.07071")
with open("to_summarize.pdf", 'wb') as f :
    f.write(doc.content)

# Read the pdf and extract text
reader = PdfReader("to_summarize.pdf")
print(f"Number of pages {len(reader.pages)}")
parts = []

def visitor_body(text, cm, tm, font_dict, font_size):
  """
  Function visitor body to tell the "extract_text" function what to do with the text.
  """
  y = tm[5]
  if 50 < y < 720 or y == 0:
      parts.append(text)

for page in reader.pages:
    page.extract_text(visitor_text=visitor_body)
text_body = "".join(parts)

Number of pages 18


## Count the number of token

In [4]:
def count_number_of_token(text:str, model:str)->int:
    #get the encoding
    tokenizer = AutoTokenizer.from_pretrained(model)  # Do not forget to set the HF_TOKEN env variable to go faster
    tokens = tokenizer.encode(text)
    number_of_token = len(tokens)
    return number_of_token

n_token = count_number_of_token(text_body, "Qwen/qwen3-8B")
print(f'Number of token : {n_token}')

Number of token : 13093


## Synthesize the document
### Run VLLM

In [6]:
import subprocess, time, requests
import os

nvidia_lib_path = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/"
os.environ['LD_LIBRARY_PATH'] = nvidia_lib_path + ":" + os.environ.get('LD_LIBRARY_PATH', '')

print(os.environ['LD_LIBRARY_PATH'])
vllm_process = subprocess.Popen([
    'vllm', 'serve',
    "Qwen/qwen3-1.7B",
    '--trust-remote-code',
    '--dtype', 'half',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.9',
], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def wait_for_server(timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        # si le process est mort, on affiche les logs et on arrête
        if vllm_process.poll() is not None:
            print(vllm_process.stdout.read())
            print(vllm_process.stderr.read())
            raise RuntimeError("Le serveur vLLM a crashé au démarrage")
        try:
            r = requests.get("http://localhost:8000/health")
            if r.status_code == 200:
                print("✅ Serveur prêt")
                return
        except requests.exceptions.ConnectionError:
            pass
        print("⏳ En attente du chargement du modèle...")
        time.sleep(5)
    raise TimeoutError("Le serveur n'a pas démarré à temps")

wait_for_server()

/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/:/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/:/usr/lib64-nvidia
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement du modèle...
⏳ En attente du chargement d

In [7]:
print(requests.get("http://localhost:8000/health").status_code)
#print(requests.get("http://localhost:8000/v1/models").json)

200


In [8]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/qwen3-1.7B","object":"model","created":1784494910,"owned_by":"vllm","root":"Qwen/qwen3-1.7B","parent":null,"max_model_len":16384,"permission":[{"id":"modelperm-a3c8f9ac64ae1af8","object":"model_permission","created":1784494910,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

### Réaliser le résumé


In [ ]:
#Méthode 1 : en définissant clairement les systèmes prompts et human prompt dans les outils de langchain approrpiés

contextual_template = "You are an helpful AI Resarcher that specializes in ML, AI and LLM papers. Please use all your expertise to approach this task. Output your content in markdown format and includes title where relevant"
system_message_prompt = SystemMessagePromptTemplate.from_template(contextual_template)
human_template = "Please summurize this paper focusing on the key important insight for each section. Expand the summary on methode so that they "
human_message_prompt = HumanMessagePromptTemplate.from_template(
    prompt = PromptTemplate(
        template=human_template,
        input_variables=["paper_content"] )
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

chat = ChatOpenAI(model_name="Qwen/qwen3-8B", base_url="http://localhost:8000/v1", api_key="not-needed")

summuray_chain = LLMChain(llm=chat, prompt=chat_prompt)
with get_openai_callback() as cb:
  summary = summuray_chain.run(paper_content=text_body)
print(summary)

In [9]:
# Méthode 2 : définir les prompts dans le chat

contextual_template = (
    "You are an helpful AI Researcher that specializes in ML, AI and LLM papers. "
    "Please use all your expertise to approach this task. Output your content in "
    "markdown format and includes title where relevant"
)

human_template = (
    "Please summarize this paper focusing on the key important insight for each "
    "section. Expand the summary on methods so that they are clearly understandable.\n\n"
    "Paper content:\n{paper_content}"
)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", contextual_template),
    ("human", human_template),
])

chat = ChatOpenAI(
    model_name="Qwen/qwen3-1.7B",
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
    extra_body={"chat_template_kwargs": {"enable_thinking": True}}
)

summary_chain = chat_prompt | chat | StrOutputParser()

summary = summary_chain.invoke({"paper_content": text_body})

print(summary)

<think>
Okay, let's start by understanding the user's request. They want a summary of the paper focusing on key insights for each section, expanding the methods to be more understandable. The paper is about the impact of generative AI on market equilibrium using a natural experiment.

First, I need to break down the paper into sections: Introduction, Related Literature, Empirical Strategy, Research Context and Data, Main Analysis, Demand Side, Supply Side, and Conclusions. For each section, I should highlight the key insights and explain the methods used.

In the Introduction, the authors discuss the emergence of generative AI and its potential to displace human creators, but they use a natural experiment to study its effects on market equilibrium. The main challenge is causal inference, so they use a natural experiment with a sudden leak of a specific AI model.

In the Related Literature section, they cover Generative AI and the economics of AI. They mention different models like GANs

In [ ]:
import os
import signal
import time

def stop_server(process, timeout=30):
    if process.poll() is not None:
        print("Le serveur est déjà arrêté.")
        return

    pgid = os.getpgid(process.pid)
    print(f"🛑 Arrêt du serveur (PID {process.pid}, PGID {pgid})...")

    # 1. SIGTERM sur tout le groupe de processus (arrêt propre)
    os.killpg(pgid, signal.SIGTERM)

    start = time.time()
    while time.time() - start < timeout:
        if process.poll() is not None:
            print("✅ Serveur arrêté proprement.")
            return
        time.sleep(1)

    # 2. Si ça ne suffit pas, on force avec SIGKILL
    print("⚠️ Le serveur ne répond pas, arrêt forcé (SIGKILL)...")
    os.killpg(pgid, signal.SIGKILL)
    process.wait()
    print("✅ Serveur tué.")

stop_server(vllm_process)